> **Disclaimer:** This is a research project. Models are not validated
> for clinical use and must not be used for medical diagnosis or
> treatment decisions. The NIH ChestX-ray14 labels were text-mined from
> radiology reports with an estimated 10–15% label noise.

# Model Evaluation on Test Set

**Purpose:** Evaluate all trained models on the held-out test set

**Models to evaluate:**
- ResNet50 (baseline)
- EfficientNet-B0
- DenseNet121

**Metrics:** AUROC per class, mean AUROC, confusion matrices, precision-recall curves

## Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import cv2
from PIL import Image
import json
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    average_precision_score,
    confusion_matrix,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
from src.dataset import ChestXrayDataset, apply_clahe, get_transforms


Device: cpu


## Configuration

In [4]:
DATA_ROOT = Path('./data')
PROCESSED_DIR = DATA_ROOT / 'processed'
MODELS_DIR = Path('./models')
RESULTS_DIR = Path('./results')
RESULTS_DIR.mkdir(exist_ok=True)

BATCH_SIZE = 32
NUM_WORKERS = 0

## Load Test Data

In [3]:
test_df = pd.read_csv(PROCESSED_DIR / 'test_df.csv')

with open(PROCESSED_DIR / 'preprocessing_config.json', 'r') as f:
    config = json.load(f)

diseases = config['diseases']
NUM_CLASSES = len(diseases)
IMG_SIZE = config['image_size']

print(f"Test samples: {len(test_df):,}")
print(f"Diseases: {diseases}")

Test samples: 25,596
Diseases: ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia']


## Dataset and DataLoader

In [ ]:
# Moved to src/dataset.py

In [ ]:
_, val_transform = get_transforms(IMG_SIZE)
test_transform = val_transform

test_dataset = ChestXrayDataset(
    test_df, diseases, transform=test_transform, return_image_id=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Test batches: {len(test_loader):,}")

## Model Definitions

In [7]:
class ResNet50Model(nn.Module):
    def __init__(self, num_classes):
        super(ResNet50Model, self).__init__()
        self.backbone = models.resnet50(weights=None)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, num_classes)
        )
    def forward(self, x):
        return self.backbone(x)

class EfficientNetB0Model(nn.Module):
    def __init__(self, num_classes):
        super(EfficientNetB0Model, self).__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        self.backbone.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        num_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, num_classes)
        )
    def forward(self, x):
        return self.backbone(x)

class DenseNet121Model(nn.Module):
    def __init__(self, num_classes):
        super(DenseNet121Model, self).__init__()
        self.backbone = models.densenet121(weights=None)
        self.backbone.features.conv0 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        num_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, num_classes)
        )
    def forward(self, x):
        return self.backbone(x)

## Evaluation Function

In [8]:
def evaluate_model(model, loader, device):
    model.eval()
    all_labels = []
    all_preds = []
    all_image_ids = []
    
    with torch.no_grad():
        for images, labels, image_ids in tqdm(loader, desc='Evaluating'):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            preds = torch.sigmoid(outputs)
            
            all_labels.append(labels.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_image_ids.extend(image_ids)
    
    all_labels = np.vstack(all_labels)
    all_preds = np.vstack(all_preds)
    
    return all_labels, all_preds, all_image_ids

## Load and Evaluate Models

In [ ]:
models_to_evaluate = [
    ('ResNet50', ResNet50Model, MODELS_DIR / 'resnet50' / 'best_model.pth'),
    ('EfficientNet-B0', EfficientNetB0Model, MODELS_DIR / 'efficientnet_b0' / 'best_model.pth'),
    ('DenseNet121', DenseNet121Model, MODELS_DIR / 'densenet121' / 'best_model.pth')
]

results = {}

for model_name, model_class, checkpoint_path in models_to_evaluate:
    if not checkpoint_path.exists():
        print(f"Skipping {model_name}: checkpoint not found")
        continue
    
    print(f"\nEvaluating {model_name}...")
    
    model = model_class(NUM_CLASSES)
    # weights_only=True is safe: checkpoints contain only tensors, strings, and lists.
    # If you see an UnpicklingError use torch.serialization.add_safe_globals([...])
    # rather than reverting to weights_only=False.
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    
    labels, preds, image_ids = evaluate_model(model, test_loader, device)
    
    per_class_auc = []
    per_class_ap = []
    
    for i in range(NUM_CLASSES):
        if len(np.unique(labels[:, i])) > 1:
            auc = roc_auc_score(labels[:, i], preds[:, i])
            ap = average_precision_score(labels[:, i], preds[:, i])
        else:
            auc = 0.0
            ap = 0.0
        per_class_auc.append(auc)
        per_class_ap.append(ap)
    
    mean_auc = np.mean(per_class_auc)
    mean_ap = np.mean(per_class_ap)
    
    results[model_name] = {
        'labels': labels,
        'predictions': preds,
        'image_ids': image_ids,
        'per_class_auc': per_class_auc,
        'per_class_ap': per_class_ap,
        'mean_auc': mean_auc,
        'mean_ap': mean_ap
    }
    
    print(f"Mean AUROC: {mean_auc:.4f}")
    print(f"Mean AP: {mean_ap:.4f}")

## Compare Models

In [ ]:
comparison_df = pd.DataFrame([
    {
        'Model': name,
        'Mean AUROC': data['mean_auc'],
        'Mean AP': data['mean_ap']
    }
    for name, data in results.items()
]).sort_values('Mean AUROC', ascending=False)

print("\nModel Comparison on Test Set:")
print(comparison_df.to_string(index=False))

comparison_df.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)

## Per-Class Performance

In [ ]:
per_class_results = []

for model_name, data in results.items():
    for disease, auc, ap in zip(diseases, data['per_class_auc'], data['per_class_ap']):
        per_class_results.append({
            'Model': model_name,
            'Disease': disease,
            'AUROC': auc,
            'AP': ap
        })

per_class_df = pd.DataFrame(per_class_results)
per_class_pivot = per_class_df.pivot(index='Disease', columns='Model', values='AUROC')

print("\nPer-Class AUROC:")
print(per_class_pivot.to_string())

per_class_pivot.to_csv(RESULTS_DIR / 'per_class_auroc.csv')

## Sensitivity, Specificity, and Clinical Metrics

In [ ]:
THRESHOLD = 0.5

sens_spec_records = []

for model_name, data in results.items():
    y_labels = data['labels']
    y_preds  = data['predictions']

    for i, disease in enumerate(diseases):
        y_true  = y_labels[:, i]
        y_score = y_preds[:, i]
        y_pred  = (y_score >= THRESHOLD).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

        sensitivity_50 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity_50 = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        # Sensitivity at 80% specificity: highest TPR where FPR <= 0.20
        if len(np.unique(y_true)) > 1:
            fpr, tpr, _ = roc_curve(y_true, y_score)
            mask = fpr <= 0.20
            sensitivity_at_80spec = float(tpr[mask].max()) if mask.any() else 0.0
        else:
            sensitivity_at_80spec = 0.0

        sens_spec_records.append({
            'Model':               model_name,
            'Disease':             disease,
            'TP':                  int(tp),
            'FP':                  int(fp),
            'FN':                  int(fn),
            'TN':                  int(tn),
            'Sensitivity@0.5':     round(sensitivity_50, 4),
            'Specificity@0.5':     round(specificity_50, 4),
            'Sensitivity@80%Spec': round(sensitivity_at_80spec, 4),
        })

sens_spec_df = pd.DataFrame(sens_spec_records)
sens_spec_df.to_csv(RESULTS_DIR / 'sensitivity_specificity.csv', index=False)

print(sens_spec_df.to_string(index=False))

## Confusion Matrices (Best Model)

In [ ]:
best_model_name = comparison_df.iloc[0]['Model']
best_data = results[best_model_name]
best_ss   = sens_spec_df[sens_spec_df['Model'] == best_model_name].set_index('Disease')

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
axes = axes.ravel()

for i, disease in enumerate(diseases):
    y_true = best_data['labels'][:, i]
    y_pred = (best_data['predictions'][:, i] >= THRESHOLD).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    sens = best_ss.loc[disease, 'Sensitivity@0.5']
    spec = best_ss.loc[disease, 'Specificity@0.5']

    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Pred Neg', 'Pred Pos'],
        yticklabels=['Actual Neg', 'Actual Pos'],
        ax=axes[i], cbar=False
    )
    axes[i].set_title(f'{disease}\nSens={sens:.3f}  Spec={spec:.3f}', fontsize=9)

for j in range(len(diseases), 16):
    axes[j].axis('off')

plt.suptitle(
    f'Confusion Matrices \u2014 {best_model_name} (threshold={THRESHOLD})',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / 'confusion_matrices_{}.png'.format(
        best_model_name.lower().replace('-', '_').replace(' ', '_')
    ),
    dpi=150, bbox_inches='tight'
)
plt.show()

## Visualization: Model Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(comparison_df))
width = 0.35

ax.bar(x - width/2, comparison_df['Mean AUROC'], width, label='AUROC', alpha=0.8)
ax.bar(x + width/2, comparison_df['Mean AP'], width, label='Average Precision', alpha=0.8)

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Comparison on Test Set')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Visualization: Per-Class AUROC Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    per_class_pivot,
    annot=True,
    fmt='.3f',
    cmap='YlGnBu',
    vmin=0.5,
    vmax=1.0,
    ax=ax,
    cbar_kws={'label': 'AUROC'}
)

ax.set_title('Per-Class AUROC by Model', fontsize=14, fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Disease')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'per_class_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## ROC Curves for Best Model

In [ ]:
best_model_name = comparison_df.iloc[0]['Model']
best_model_data = results[best_model_name]

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.ravel()

for i, disease in enumerate(diseases):
    if i < len(axes):
        ax = axes[i]
        
        labels = best_model_data['labels'][:, i]
        preds = best_model_data['predictions'][:, i]
        
        if len(np.unique(labels)) > 1:
            fpr, tpr, _ = roc_curve(labels, preds)
            auc = best_model_data['per_class_auc'][i]
            
            ax.plot(fpr, tpr, label=f'AUROC = {auc:.3f}', linewidth=2)
            ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
            ax.set_xlabel('False Positive Rate')
            ax.set_ylabel('True Positive Rate')
            ax.set_title(disease)
            ax.legend(loc='lower right')
            ax.grid(True, alpha=0.3)

if len(diseases) < len(axes):
    axes[-1].axis('off')

plt.suptitle(f'ROC Curves - {best_model_name}', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'roc_curves_{best_model_name.lower().replace("-", "_")}.png', 
            dpi=300, bbox_inches='tight')
plt.show()

## Save Results

In [ ]:
import datetime

best_model_name = comparison_df.iloc[0]['Model']
ss_lookup = sens_spec_df.set_index(['Model', 'Disease'])

final_results = {
    'models': {},
    'best_model': best_model_name,
    'threshold_default': THRESHOLD,
    'evaluation_date': datetime.date.today().isoformat(),
}

for model_name, data in results.items():
    per_class = {}
    for i, disease in enumerate(diseases):
        row = ss_lookup.loc[(model_name, disease)]
        per_class[disease] = {
            'auroc':                 float(data['per_class_auc'][i]),
            'ap':                    float(data['per_class_ap'][i]),
            'sensitivity_at_50':     float(row['Sensitivity@0.5']),
            'specificity_at_50':     float(row['Specificity@0.5']),
            'sensitivity_at_80spec': float(row['Sensitivity@80%Spec']),
        }
    final_results['models'][model_name] = {
        'mean_auroc': float(data['mean_auc']),
        'per_class':  per_class,
    }

RESULTS_DIR.mkdir(exist_ok=True)
with open(RESULTS_DIR / 'test_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print(f"Saved to {RESULTS_DIR / 'test_results.json'}")